<a href="https://colab.research.google.com/github/tkoganti/tkoganti.github.io/blob/master/custom_variant_scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import zipfile
import os
from tqdm import tqdm
from alphagenome.data import genome
from alphagenome_research.model import dna_model
from alphagenome.models import variant_scorers


from google.colab import files

# Upload your variant text file
uploaded = files.upload()
variants_file = list(uploaded.keys())[0]
print(f"Uploaded: {variants_file}")

# Upload your checkpoint zip
print("\nUpload your checkpoint zip...")
uploaded = files.upload()
checkpoint_zip = list(uploaded.keys())[0]
print(f"Uploaded: {checkpoint_zip}")

/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.27.2 is exactly one major version older than the runtime version 6.31.1 at alphagenome/protos/dna_model.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.27.2 is exactly one major version older than the runtime version 6.31.1 at alphagenome/protos/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  wa

Saving IRF4_var_AGinput.tsv to IRF4_var_AGinput.tsv
Uploaded: IRF4_var_AGinput.tsv

Upload your checkpoint zip...


Saving checkpoint.zip to checkpoint.zip
Uploaded: checkpoint.zip


In [4]:
# Upgrade protobuf to match gencode version
!pip install protobuf==6.31.1
print("Done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 kB 29.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.31.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.31.1 which is incompatible.


Done!


In [1]:

from IPython.display import clear_output

!PIP_NO_BINARY=pyBigWig pip install \
    git+https://github.com/google-deepmind/alphagenome_research.git


print("AlphaGenome installed!")

# Verify
import alphagenome
import alphagenome_research
print("alphagenome version:", alphagenome.__version__)

  Cloning https://github.com/google-deepmind/alphagenome_research.git to /tmp/pip-req-build-qzvochsp
  Running command git clone --filter=blob:none --quiet https://github.com/google-deepmind/alphagenome_research.git /tmp/pip-req-build-qzvochsp
  Resolved https://github.com/google-deepmind/alphagenome_research.git to commit dad09dd11a480fb8c16ae703a606f55a6e3e968b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
AlphaGenome installed!
alphagenome version: 0.6.1


In [2]:
# Extract checkpoint model
print("\nExtracting checkpoint...")
os.makedirs('/content/checkpoint_mm', exist_ok=True)
with zipfile.ZipFile(f'/content/{checkpoint_zip}', 'r') as z:
    z.extractall('/content/checkpoint_mm/')
print("Extracted!")
!ls /content/checkpoint_mm/


Extracting checkpoint...
Extracted!
array_metadatas       d		      _METADATA        _sharding
_CHECKPOINT_METADATA  manifest.ocdbt  ocdbt.process_0


In [21]:
# Apply splice patch
filepath = '/usr/local/lib/python3.12/dist-packages/alphagenome_research/model/dna_model.py'

with open(filepath, 'r') as f:
    content = f.read()

old_code = """  reference_splice_sites = (
      reference_predictions['splice_sites_classification']['predictions']
      * splice_junction_masks.reference_genes
  )
  alternate_splice_sites = alternate_predictions['splice_sites_classification'][
      'predictions'
  ]"""

new_code = """  if 'splice_sites_classification' in reference_predictions:
    reference_splice_sites = (
        reference_predictions['splice_sites_classification']['predictions']
        * splice_junction_masks.reference_genes
    )
    alternate_splice_sites = alternate_predictions['splice_sites_classification'][
        'predictions'
    ]
  else:
        reference_splice_sites = None
    alternate_splice_sites = None"""

new_content = content.replace(old_code, new_code)
if new_content != content:
    with open(filepath, 'w') as f:
        f.write(new_content)
    print("✅ Splice patch applied!")
else:
    print("Already patched!")


Already patched!


In [4]:
# load variants
df_variants = pd.read_csv(f'/content/{variants_file}', sep='\t')
print(f"Loaded {len(df_variants)} variants")
print(df_variants.head())

# Use only first 4 columns
vcf = pd.DataFrame({
    'variant_id': df_variants['chrom'].astype(str) + '_' +
                  df_variants['pos'].astype(str) + '_' +
                  df_variants['ref'] + '_' +
                  df_variants['alt'],
    'CHROM': df_variants['chrom'].apply(
        lambda x: x if str(x).startswith('chr') else f'chr{x}'
    ),
    'POS': df_variants['pos'].astype(int),
    'REF': df_variants['ref'],
    'ALT': df_variants['alt'],
})
print(f"\nFormatted {len(vcf)} variants")
print(vcf.head())

Loaded 70 variants
  chrom     pos ref alt sample_name                                   gene
0  chr6  229915   T   C    BM068331  416J7.5-DUSP22,DUSP22,RP3,RP3-416J7.5
1  chr6  301351   G   A    BM072030                                 DUSP22
2  chr6  256528   T   C    BM042351  416J7.5-DUSP22,DUSP22,RP3,RP3-416J7.5
3  chr6  313674   G   T    BM062926                                 DUSP22
4  chr6  362066   C   T    BM049787                                   IRF4

Formatted 70 variants
        variant_id CHROM     POS REF ALT
0  chr6_229915_T_C  chr6  229915   T   C
1  chr6_301351_G_A  chr6  301351   G   A
2  chr6_256528_T_C  chr6  256528   T   C
3  chr6_313674_G_T  chr6  313674   G   T
4  chr6_362066_C_T  chr6  362066   C   T


In [5]:
uploaded = files.upload()
track_metadata_file = list(uploaded.keys())[0]
print(f"Uploaded: {track_metadata_file}")

# Load it
TRACK_METADATA = pd.read_csv(f'/content/{track_metadata_file}')
print("TRACK_METADATA loaded:")
print(TRACK_METADATA)

Saving track_metadata.csv to track_metadata.csv
Uploaded: track_metadata.csv
TRACK_METADATA loaded:
   output_type                                               name strand  \
0      RNA_SEQ  MM_BM061418.Aligned.sortedByCoord.out total RN...      .   
1      RNA_SEQ  MM_BM061472.Aligned.sortedByCoord.out total RN...      .   
2      RNA_SEQ  MM_BM63568R.Aligned.sortedByCoord.out total RN...      .   
3      RNA_SEQ  MM_BM64752R.Aligned.sortedByCoord.out total RN...      .   
4      RNA_SEQ  MM_BM65549R.Aligned.sortedByCoord.out total RN...      .   
5      RNA_SEQ  MM_BM65562R.Aligned.sortedByCoord.out total RN...      .   
6      RNA_SEQ  MM_BM65678R.Aligned.sortedByCoord.out total RN...      .   
7      RNA_SEQ  MM_BM65904R.Aligned.sortedByCoord.out total RN...      .   
8      RNA_SEQ  MM_BM66361R.Aligned.sortedByCoord.out total RN...      .   
9      RNA_SEQ  MM_BM67614R.Aligned.sortedByCoord.out total RN...      .   
10     RNA_SEQ  MM_BM68182R.Aligned.sortedByCoord.out total RN..

In [6]:
# Rebuild TRACK METADATA
TRACK_METADATA = pd.read_csv('/content/track_metadata.csv')

print("\nTrack metadata:")
print(TRACK_METADATA)



# Rebuild output metadata
import dataclasses
import orbax.checkpoint as ocp
from alphagenome_research.model.metadata import metadata as metadata_lib


def build_output_metadata(track_metadata):
    metadata = {}
    for output_type, df_group in track_metadata.groupby('output_type'):
        output_type_enum = dna_model.OutputType[str(output_type)]
        metadata[output_type_enum.name.lower()] = df_group
    return metadata_lib.AlphaGenomeOutputMetadata(**metadata)

output_metadata = {
    dna_model.Organism.HOMO_SAPIENS: build_output_metadata(TRACK_METADATA)
}
print("Output metadata built!")


Track metadata:
   output_type                                               name strand  \
0      RNA_SEQ  MM_BM061418.Aligned.sortedByCoord.out total RN...      .   
1      RNA_SEQ  MM_BM061472.Aligned.sortedByCoord.out total RN...      .   
2      RNA_SEQ  MM_BM63568R.Aligned.sortedByCoord.out total RN...      .   
3      RNA_SEQ  MM_BM64752R.Aligned.sortedByCoord.out total RN...      .   
4      RNA_SEQ  MM_BM65549R.Aligned.sortedByCoord.out total RN...      .   
5      RNA_SEQ  MM_BM65562R.Aligned.sortedByCoord.out total RN...      .   
6      RNA_SEQ  MM_BM65678R.Aligned.sortedByCoord.out total RN...      .   
7      RNA_SEQ  MM_BM65904R.Aligned.sortedByCoord.out total RN...      .   
8      RNA_SEQ  MM_BM66361R.Aligned.sortedByCoord.out total RN...      .   
9      RNA_SEQ  MM_BM67614R.Aligned.sortedByCoord.out total RN...      .   
10     RNA_SEQ  MM_BM68182R.Aligned.sortedByCoord.out total RN...      .   
11     RNA_SEQ  MM_BM68266R.Aligned.sortedByCoord.out total RN...      

In [7]:
from alphagenome_research.model.metadata import metadata as metadata_lib
print(dir(metadata_lib))

['AlphaGenomeOutputMetadata', 'Bool', 'Collection', 'Int32', 'Mapping', '_PADDING_TRACK_NAME', '_PATH_METADATA', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_create_output_strand_reindexing', 'create_track_masks', 'dataclasses', 'dna_client', 'dna_model', 'dna_model_service_pb2', 'dna_output', 'functools', 'load', 'np', 'ontology', 'pathlib', 'resources', 'text_format', 'track_data', 'typing']


In [8]:

from alphagenome_research.finetuning import finetune

# metadata is nested
from alphagenome_research.model.metadata import metadata as metadata_lib

# Check AlphaGenomeOutputMetadata
print([m for m in dir(metadata_lib) if not m.startswith('_')])

['AlphaGenomeOutputMetadata', 'Bool', 'Collection', 'Int32', 'Mapping', 'create_track_masks', 'dataclasses', 'dna_client', 'dna_model', 'dna_model_service_pb2', 'dna_output', 'functools', 'load', 'np', 'ontology', 'pathlib', 'resources', 'text_format', 'track_data', 'typing']


In [11]:
# Extract finetuned model
import os
import zipfile

os.makedirs('/content/checkpoint_mm', exist_ok=True)
with zipfile.ZipFile('/content/checkpoint.zip', 'r') as z:
    z.extractall('/content/checkpoint_mm/')

print("Extracted!")
!ls /content/checkpoint_mm/

Extracted!
array_metadatas       d		      _METADATA        _sharding
_CHECKPOINT_METADATA  manifest.ocdbt  ocdbt.process_0


In [15]:
# Extract base model
import os

# Extract all_folds tar.gz
os.makedirs('/content/alphagenome_all_folds', exist_ok=True)
print("Extracting all_folds weights...")
!tar -xzf /content/alphagenome-jax-all_folds-v1.tar.gz \
    -C /content/alphagenome_all_folds/
print("Done!")
!ls /content/alphagenome_all_folds/

Extracting all_folds weights...
Done!
_CHECKPOINT_METADATA  d  manifest.ocdbt  _METADATA  ocdbt.process_0


In [36]:
# Load the model
import os

# Verify checkpoint_mm extracted correctly
print("checkpoint_mm contents:")
!ls /content/checkpoint_mm/

# Config
SEQUENCE_LENGTH = int(2**17)
ORGANISM = dna_model.Organism.HOMO_SAPIENS

# Check ModelVersion options
print("\nAvailable ModelVersions:")
print([m.name for m in dna_model.ModelVersion])

# Load base weights
checkpointer = ocp.StandardCheckpointer()
params_base, state_base = checkpointer.restore(
    '/content/alphagenome_all_folds/'
)
print("✅ Base weights loaded!")

# Create fine-tuned model
default_settings_human = dna_model.default_organism_settings()[
    dna_model.Organism.HOMO_SAPIENS
]
settings_human_finetune = dataclasses.replace(
    default_settings_human,
    metadata=output_metadata[dna_model.Organism.HOMO_SAPIENS],
)
model = dna_model.create(
    '/content/checkpoint_mm/',
    organism_settings={
        dna_model.Organism.HOMO_SAPIENS: settings_human_finetune
    },
)
print("✅ Fine-tuned model loaded!")



checkpoint_mm contents:
array_metadatas       d		      _METADATA        _sharding
_CHECKPOINT_METADATA  manifest.ocdbt  ocdbt.process_0

Available ModelVersions:
['ALL_FOLDS', 'FOLD_0', 'FOLD_1', 'FOLD_2', 'FOLD_3']
✅ Base weights loaded!


/usr/local/lib/python3.12/dist-packages/pyfaidx/__init__.py:596: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


✅ Fine-tuned model loaded!


In [37]:
# Run variant scoring
from tqdm import tqdm
from alphagenome.models import variant_scorers

results = []
failed = []

for i, vcf_row in tqdm(vcf.iterrows(), total=len(vcf)):
    try:
        variant = genome.Variant(
            chromosome=str(vcf_row.CHROM),
            position=int(vcf_row.POS),
            reference_bases=vcf_row.REF,
            alternate_bases=vcf_row.ALT,
            name=str(vcf_row.variant_id),
        )
        interval = variant.reference_interval.resize(SEQUENCE_LENGTH)

        variant_scores = model.score_variant(
            interval=interval,
            variant=variant,
            variant_scorers=[
                variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']
            ],
            organism=dna_model.Organism.HOMO_SAPIENS,
        )
        results.append(variant_scores)

    except Exception as e:
        print(f"Failed: {vcf_row.variant_id}: {e}")
        failed.append(vcf_row.variant_id)

print(f"\n✅ Scored: {len(results)} variants")
print(f"❌ Failed: {len(failed)}")

# Convert to dataframe
df_scores = variant_scorers.tidy_scores(results)
print(df_scores.head(10))

# Save
df_scores.to_csv(
    '/content/drive/MyDrive/mm_finetune/mm_variant_scores.csv',
    index=False
)
from google.colab import files
files.download('/content/drive/MyDrive/mm_finetune/mm_variant_scores.csv')
print("✅ Saved and downloaded!")

  1%|▏         | 1/70 [00:46<53:55, 46.89s/it]

Failed: chr6_229915_T_C: Can only compare identically-labeled (both index and columns) DataFrame objects


  4%|▍         | 3/70 [00:47<11:55, 10.67s/it]

Failed: chr6_301351_G_A: Can only compare identically-labeled (both index and columns) DataFrame objects
Failed: chr6_256528_T_C: Can only compare identically-labeled (both index and columns) DataFrame objects


  7%|▋         | 5/70 [00:47<04:35,  4.24s/it]

...

 99%|█████████▊| 69/70 [01:01<00:00,  5.30it/s]

Failed: chr6_300358_G_T: Can only compare identically-labeled (both index and columns) DataFrame objects


100%|██████████| 70/70 [01:01<00:00,  1.13it/s]

Failed: chr6_319139_A_T: Can only compare identically-labeled (both index and columns) DataFrame objects

✅ Scored: 0 variants
❌ Failed: 70


AttributeError: 'NoneType' object has no attribute 'head'

In [26]:
# Patch that works for splice classification
filepath = '/usr/local/lib/python3.12/dist-packages/alphagenome_research/model/dna_model.py'

with open(filepath, 'r') as f:
    content = f.read()

# Fix the indentation issue
old_code = """  if 'splice_sites_classification' in reference_predictions:
    reference_splice_sites = (
        reference_predictions['splice_sites_classification']['predictions']
        * splice_junction_masks.reference_genes
    )
    alternate_splice_sites = alternate_predictions['splice_sites_classification'][
        'predictions'
    ]
  else:
        reference_splice_sites = None
    alternate_splice_sites = None"""

new_code = """  if 'splice_sites_classification' in reference_predictions:
    reference_splice_sites = (
        reference_predictions['splice_sites_classification']['predictions']
        * splice_junction_masks.reference_genes
    )
    alternate_splice_sites = alternate_predictions['splice_sites_classification'][
        'predictions'
    ]
  else:
    reference_splice_sites = None
    alternate_splice_sites = None"""

new_content = content.replace(old_code, new_code)

if new_content == content:
    print("❌ Pattern not found")
else:
    with open(filepath, 'w') as f:
        f.write(new_content)
    print("✅ Indentation fixed!")

# Verify
lines = new_content.split('\n')
for i, line in enumerate(lines[218:238], start=219):
    print(f"{i}: {repr(line)}")

✅ Indentation fixed!
219: '      params,'
220: '      state,'
221: '      alternate_sequences,'
222: '      organism_indices,'
223: '  )'
224: "  if 'splice_sites_classification' in reference_predictions:"
225: '    reference_splice_sites = ('
226: "        reference_predictions['splice_sites_classification']['predictions']"
227: '        * splice_junction_masks.reference_genes'
228: '    )'
229: "    alternate_splice_sites = alternate_predictions['splice_sites_classification']["
230: "        'predictions'"
231: '    ]'
232: '  else:'
233: '    reference_splice_sites = None'
234: '    alternate_splice_sites = None'
235: ''
236: "  reference_trunk = reference_predictions['embeddings_1bp']"
237: "  alternate_trunk = alternate_predictions['embeddings_1bp']"
238: ''


In [35]:
import importlib
import alphagenome_research.model.dna_model as dna_model_module
importlib.reload(dna_model_module)
print("✅ Module reloaded!")

✅ Module reloaded!


In [32]:
filepath = '/usr/local/lib/python3.12/dist-packages/alphagenome_research/model/dna_model.py'

with open(filepath, 'r') as f:
    content = f.read()

old_code = """  # Get union of splice site positions across ref and alt.
  ref_and_alt_splice_site_positions = splicing.generate_splice_site_positions(
      ref=reference_splice_sites,
      alt=alternate_splice_sites,
      splice_sites=splice_junction_masks.splice_sites,
      k=num_splice_sites,
      pad_to_length=num_splice_sites,
      threshold=splice_site_threshold,
  )"""

new_code = """  # Get union of splice site positions across ref and alt.
  if reference_splice_sites is None or alternate_splice_sites is None:
    ref_and_alt_splice_site_positions = None
  else:
    ref_and_alt_splice_site_positions = splicing.generate_splice_site_positions(
        ref=reference_splice_sites,
        alt=alternate_splice_sites,
        splice_sites=splice_junction_masks.splice_sites,
        k=num_splice_sites,
        pad_to_length=num_splice_sites,
        threshold=splice_site_threshold,
    )"""

new_content = content.replace(old_code, new_code)

if new_content == content:
    print("❌ Pattern not found")
else:
    with open(filepath, 'w') as f:
        f.write(new_content)
    print("✅ Second patch applied!")

# Verify
lines = new_content.split('\n')
for i, line in enumerate(lines[255:275], start=256):
    print(f"{i}: {repr(line)}")

✅ Second patch applied!
256: '    )'
257: ''
258: '  # Get union of splice site positions across ref and alt.'
259: '  if reference_splice_sites is None or alternate_splice_sites is None:'
260: '    ref_and_alt_splice_site_positions = None'
261: '  else:'
262: '    ref_and_alt_splice_site_positions = splicing.generate_splice_site_positions('
263: '        ref=reference_splice_sites,'
264: '        alt=alternate_splice_sites,'
265: '        splice_sites=splice_junction_masks.splice_sites,'
266: '        k=num_splice_sites,'
267: '        pad_to_length=num_splice_sites,'
268: '        threshold=splice_site_threshold,'
269: '    )'
270: "  reference_predictions['splice_sites_junction'] = junctions_apply_fn("
271: '      params,'
272: '      state,'
273: '      reference_trunk,'
274: '      ref_and_alt_splice_site_positions,'
275: '      organism_indices,'


In [34]:
filepath = '/usr/local/lib/python3.12/dist-packages/alphagenome_research/model/dna_model.py'

with open(filepath, 'r') as f:
    content = f.read()

old_code = """  reference_predictions['splice_sites_junction'] = junctions_apply_fn(
      params,
      state,
      reference_trunk,
      ref_and_alt_splice_site_positions,
      organism_indices,
  )
  alternate_predictions['splice_sites_junction'] = junctions_apply_fn(
      params,
      state,
      alternate_trunk,
      ref_and_alt_splice_site_positions,
      organism_indices,
  )"""

new_code = """  if ref_and_alt_splice_site_positions is not None:
    reference_predictions['splice_sites_junction'] = junctions_apply_fn(
        params,
        state,
        reference_trunk,
        ref_and_alt_splice_site_positions,
        organism_indices,
    )
    alternate_predictions['splice_sites_junction'] = junctions_apply_fn(
        params,
        state,
        alternate_trunk,
        ref_and_alt_splice_site_positions,
        organism_indices,
    )"""

new_content = content.replace(old_code, new_code)

if new_content == content:
    print("❌ Pattern not found")
else:
    with open(filepath, 'w') as f:
        f.write(new_content)
    print("✅ Third patch applied!")

# Verify
lines = new_content.split('\n')
for i, line in enumerate(lines[268:300], start=269):
    print(f"{i}: {repr(line)}")

✅ Third patch applied!
269: '    )'
270: '  if ref_and_alt_splice_site_positions is not None:'
271: "    reference_predictions['splice_sites_junction'] = junctions_apply_fn("
272: '        params,'
273: '        state,'
274: '        reference_trunk,'
275: '        ref_and_alt_splice_site_positions,'
276: '        organism_indices,'
277: '    )'
278: "    alternate_predictions['splice_sites_junction'] = junctions_apply_fn("
279: '        params,'
280: '        state,'
281: '        alternate_trunk,'
282: '        ref_and_alt_splice_site_positions,'
283: '        organism_indices,'
284: '    )'
285: ''
286: '  def _extract_and_rc(predictions):'
287: '    return augmentation.reverse_complement('
288: '        extract_predictions(predictions, requested_outputs),'
289: '        negative_strand_mask,'
290: '        strand_reindexing=strand_reindexing,'
291: '        sequence_length=sequence_length,'
292: '    )'
293: ''
294: '  return ('
295: '      _extract_and_rc(reference_predictions),'


In [38]:
results = []
failed = []

for i, vcf_row in tqdm(vcf.iterrows(), total=len(vcf)):
    try:
        variant = genome.Variant(
            chromosome=str(vcf_row.CHROM),
            position=int(vcf_row.POS),
            reference_bases=vcf_row.REF,
            alternate_bases=vcf_row.ALT,
            name=str(vcf_row.variant_id),
        )
        interval = variant.reference_interval.resize(SEQUENCE_LENGTH)

        variant_output = model.score_variant(
            interval=interval,
            variant=variant,
            variant_scorers=[
                variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']
            ],
            organism=dna_model.Organism.HOMO_SAPIENS,
        )

        # Extract scores manually instead of using tidy_scores
        for scorer_name, score_df in variant_output.items():
            if score_df is not None:
                results.append({
                    'variant_id': vcf_row.variant_id,
                    'chromosome': vcf_row.CHROM,
                    'position': vcf_row.POS,
                    'ref': vcf_row.REF,
                    'alt': vcf_row.ALT,
                    'scorer': str(scorer_name),
                    'raw_score': float(score_df) if hasattr(score_df, '__float__') else str(score_df),
                })

    except Exception as e:
        print(f"Failed: {vcf_row.variant_id}: {e}")
        failed.append(vcf_row.variant_id)

print(f"\n✅ Scored: {len(results)} variants")
print(f"❌ Failed: {len(failed)}")

  1%|▏         | 1/70 [00:00<00:12,  5.36it/s]

Failed: chr6_229915_T_C: Can only compare identically-labeled (both index and columns) DataFrame objects


  4%|▍         | 3/70 [00:00<00:13,  5.11it/s]

Failed: chr6_301351_G_A: Can only compare identically-labeled (both index and columns) DataFrame objects
Failed: chr6_256528_T_C: Can only compare identically-labeled (both index and columns) DataFrame objects


  7%|▋         | 5/70 [00:00<00:12,  5.31it/s]

...

Failed: chr6_279113_T_C: Can only compare identically-labeled (both index and columns) DataFrame objects
Failed: chr6_375213_C_T: Can only compare identically-labeled (both index and columns) DataFrame objects


 99%|█████████▊| 69/70 [00:12<00:00,  5.34it/s]

Failed: chr6_300358_G_T: Can only compare identically-labeled (both index and columns) DataFrame objects


100%|██████████| 70/70 [00:13<00:00,  5.34it/s]

Failed: chr6_319139_A_T: Can only compare identically-labeled (both index and columns) DataFrame objects

✅ Scored: 0 variants
❌ Failed: 70


In [43]:
# Check what scoring models are available
from alphagenome.models import variant_scorers

# Check available scorer classes
print([x for x in dir(variant_scorers) if not x.startswith('_')])

['AggregationType', 'BaseVariantScorer', 'CenterMaskScorer', 'ContactMapScorer', 'GeneMaskActiveScorer', 'GeneMaskLFCScorer', 'GeneMaskSplicingScorer', 'PolyadenylationScorer', 'RECOMMENDED_VARIANT_SCORERS', 'SUPPORTED_AGGREGATIONS', 'SUPPORTED_ORGANISMS', 'SUPPORTED_OUTPUT_TYPES', 'SUPPORTED_WIDTHS', 'Sequence', 'SpliceJunctionScorer', 'TypeVar', 'VariantScorerTypes', 'anndata', 'dataclasses', 'dna_model_pb2', 'dna_output', 'enum', 'get_recommended_scorers', 'immutabledict', 'itertools', 'math', 'pd', 'tidy_anndata', 'tidy_scores']


In [44]:
from alphagenome.models.variant_scorers import CenterMaskScorer
from alphagenome.models import dna_output

# Create simple center mask scorer - no gene annotations needed!
simple_scorer = CenterMaskScorer(
    requested_output=dna_output.OutputType.RNA_SEQ,
    width=10000
)
print("Scorer:", simple_scorer)

# Retry scoring
results = []
failed = []

for i, vcf_row in tqdm(vcf.iterrows(), total=len(vcf)):
    try:
        variant = genome.Variant(
            chromosome=str(vcf_row.CHROM),
            position=int(vcf_row.POS),
            reference_bases=vcf_row.REF,
            alternate_bases=vcf_row.ALT,
            name=str(vcf_row.variant_id),
        )
        interval = variant.reference_interval.resize(SEQUENCE_LENGTH)

        variant_scores = model.score_variant(
            interval=interval,
            variant=variant,
            variant_scorers=[simple_scorer],
            organism=dna_model.Organism.HOMO_SAPIENS,
        )
        results.append(variant_scores)

    except Exception as e:
        print(f"Failed: {vcf_row.variant_id}: {e}")
        failed.append(vcf_row.variant_id)

print(f"\n✅ Scored: {len(results)} variants")
print(f"❌ Failed: {len(failed)}")

# Convert to dataframe
if results:
    df_scores = variant_scorers.tidy_scores(results)
    print(df_scores.head(10))

TypeError: CenterMaskScorer.__init__() missing 1 required positional argument: 'aggregation_type'

In [45]:
from alphagenome.models.variant_scorers import CenterMaskScorer, AggregationType
from alphagenome.models import dna_output

# Check available aggregation types
print("Aggregation types:", [a.name for a in AggregationType])

Aggregation types: ['DIFF_MEAN', 'DIFF_SUM', 'DIFF_SUM_LOG2', 'DIFF_LOG2_SUM', 'L2_DIFF', 'L2_DIFF_LOG1P', 'ACTIVE_MEAN', 'ACTIVE_SUM']


In [49]:
from alphagenome.models.variant_scorers import CenterMaskScorer, AggregationType
from alphagenome.models import dna_output

simple_scorer = CenterMaskScorer(
    requested_output=dna_output.OutputType.RNA_SEQ,
    width=10001,  # ← supported width
    aggregation_type=AggregationType.DIFF_MEAN
)
print("Scorer:", simple_scorer)

# Retry scoring
results = []
failed = []

for i, vcf_row in tqdm(vcf.iterrows(), total=len(vcf)):
    try:
        variant = genome.Variant(
            chromosome=str(vcf_row.CHROM),
            position=int(vcf_row.POS),
            reference_bases=vcf_row.REF,
            alternate_bases=vcf_row.ALT,
            name=str(vcf_row.variant_id),
        )
        interval = variant.reference_interval.resize(SEQUENCE_LENGTH)

        variant_scores = model.score_variant(
            interval=interval,
            variant=variant,
            variant_scorers=[simple_scorer],
            organism=dna_model.Organism.HOMO_SAPIENS,
        )
        results.append(variant_scores)

    except Exception as e:
        print(f"Failed: {vcf_row.variant_id}: {e}")
        failed.append(vcf_row.variant_id)

print(f"\n✅ Scored: {len(results)} variants")
print(f"❌ Failed: {len(failed)}")

if results:
    df_scores = variant_scorers.tidy_scores(results)
    print(df_scores.head(10))
    print("\nColumns:", df_scores.columns.tolist())

Scorer: CenterMaskScorer(requested_output=RNA_SEQ, width=10001, aggregation_type=DIFF_MEAN)


100%|██████████| 70/70 [00:10<00:00,  6.60it/s]



✅ Scored: 70 variants
❌ Failed: 0
        variant_id       scored_interval gene_id gene_name gene_type  \
0  chr6:229915:T>C  chr6:164379-295451:.    None      None      None   
1  chr6:229915:T>C  chr6:164379-295451:.    None      None      None   
2  chr6:229915:T>C  chr6:164379-295451:.    None      None      None   
3  chr6:229915:T>C  chr6:164379-295451:.    None      None      None   
4  chr6:229915:T>C  chr6:164379-295451:.    None      None      None   
5  chr6:229915:T>C  chr6:164379-295451:.    None      None      None   
6  chr6:229915:T>C  chr6:164379-295451:.    None      None      None   
7  chr6:229915:T>C  chr6:164379-295451:.    None      None      None   
8  chr6:229915:T>C  chr6:164379-295451:.    None      None      None   
9  chr6:229915:T>C  chr6:164379-295451:.    None      None      None   

  gene_strand junction_Start junction_End output_type  \
0        None           None         None     RNA_SEQ   
1        None           None         None     RNA_SEQ   
2

In [53]:
# Save and download
df_scores.to_csv('/content/variant_scores.csv', index=False)

from google.colab import files
files.download('/content/variant_scores.csv')
print("✅ Downloaded!")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded!


In [54]:
import numpy as np

# Add quantile score manually
df_scores['quantile_score'] = df_scores.groupby(
    'track_name')['raw_score'].rank(pct=True)

print("Columns now:", df_scores.columns.tolist())
print(f"\nTotal variants: {len(df_scores)}")
print(f"\nTop 10 by abs raw_score:")
print(df_scores.reindex(
    df_scores['raw_score'].abs().sort_values(ascending=False).index
).head(10)[['variant_id', 'track_name', 'raw_score', 'quantile_score']])

# Save and download
df_scores.to_csv('/content/variant_scores.csv', index=False)

from google.colab import files
files.download('/content/variant_scores.csv')
print("\n✅ Downloaded!")



Columns now: ['variant_id', 'scored_interval', 'gene_id', 'gene_name', 'gene_type', 'gene_strand', 'junction_Start', 'junction_End', 'output_type', 'variant_scorer', 'track_name', 'track_strand', 'raw_score', 'quantile_score']

Total variants: 1050

Top 10 by abs raw_score:
          variant_id                                         track_name  \
965  chr6:347370:G>A  MM_BM65562R.Aligned.sortedByCoord.out total RN...   
498  chr6:349711:C>T  MM_BM64752R.Aligned.sortedByCoord.out total RN...   
974  chr6:347370:G>A                       MM_chr_R072130 total RNA-seq   
723  chr6:347816:G>A  MM_BM64752R.Aligned.sortedByCoord.out total RN...   
973  chr6:347370:G>A                       MM_chr_R050842 total RNA-seq   
969  chr6:347370:G>A  MM_BM67614R.Aligned.sortedByCoord.out total RN...   
733  chr6:347816:G>A                       MM_chr_R050842 total RNA-seq   
972  chr6:347370:G>A  MM_BM68272R.Aligned.sortedByCoord.out total RN...   
962  chr6:347370:G>A  MM_BM63568R.Aligned.sortedBy

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Downloaded!


OSError: Cannot save file into a non-existent directory: '/content/drive/MyDrive/mm_finetune'